<a href="https://colab.research.google.com/github/Ron-Maina/Brain-Tumour-Segmentation-/blob/main/Multi_Region_Brain_Tumour_Segmentation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# IMPORT KAGGLE DATA SOURCES
import kagglehub
medical_segmentation_decathlon_brain_tumour_path = kagglehub.dataset_download('thisisrick25/medical-segmentation-decathlon-brain-tumour')

print('Data source import complete.')


100%|██████████| 7.39G/7.39G [01:25<00:00, 93.2MB/s]

Extracting files...


KeyboardInterrupt: 

In [ ]:
!pip install -q "monai[nibabel]"

import os
import re
import json
import time

import numpy as np
import nibabel as nib
import matplotlib.pyplot as plt

import torch
from sklearn.model_selection import train_test_split

from monai.data import Dataset, DataLoader
from monai.transforms import (
    Compose, LoadImaged, EnsureChannelFirstd, NormalizeIntensityd,
    RandCropByPosNegLabeld, RandFlipd, EnsureTyped,
)
from monai.networks.nets import UNet, AttentionUnet
from monai.losses import DiceCELoss
from monai.metrics import DiceMetric
from monai.inferers import sliding_window_inference

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

In [ ]:
# GPU check
print("GPU available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU name:", torch.cuda.get_device_name(0))

BASE_DIR = medical_segmentation_decathlon_brain_tumour_path
for root, dirs, files in os.walk(BASE_DIR):
    depth = root.replace(BASE_DIR, "").count(os.sep)
    if depth > 2:
        continue
    print("  " * depth + os.path.basename(root) + "/")

# should contain imagesTr/ and labelsTr/
DATA_DIR = os.path.join(BASE_DIR, "Task01_BrainTumour")

# checkpoint save/reload sanity check
dummy = {"epoch": 0, "state_dict": {}}
torch.save(dummy, "/kaggle/working/checkpoint_test.pt")
reloaded = torch.load("/kaggle/working/checkpoint_test.pt")
print("Checkpoint reload OK:", reloaded["epoch"] == 0)

### Data Exploration

In [ ]:
import os
import re
import nibabel as nib
import numpy as np

IMAGES_DIR = os.path.join(DATA_DIR, "imagesTr")
LABELS_DIR = os.path.join(DATA_DIR, "labelsTr")

def is_real_file(filename):
    # skip "._" junk files
    return not filename.startswith("._")

def patient_id(filename):
    match = re.match(r"(BRATS_\d+)", filename)
    return match.group(1) if match else None

image_files = {patient_id(f): os.path.join(IMAGES_DIR, f)
               for f in os.listdir(IMAGES_DIR) if is_real_file(f)}
label_files = {patient_id(f): os.path.join(LABELS_DIR, f)
               for f in os.listdir(LABELS_DIR) if is_real_file(f)}
# matched by patient ID, not filename

patient_ids = sorted(set(image_files) & set(label_files))
print(f"Found {len(patient_ids)} patients with both an image and a label file")

missing_labels = set(image_files) - set(label_files)
missing_images = set(label_files) - set(image_files)
if missing_labels:
    print("Image but no label:", sorted(missing_labels))
if missing_images:
    print("Label but no image:", sorted(missing_images))

issues = []
for pid in patient_ids:
    try:
        image = nib.load(image_files[pid])
        label = nib.load(label_files[pid])
    except Exception as error:
        issues.append((pid, f"failed to load: {error}"))
        continue

    if image.shape[-1] != 4:
        issues.append((pid, f"expected 4 channels, got shape {image.shape}"))

    label_classes = set(np.unique(label.get_fdata()))
    if not label_classes.issubset({0, 1, 2, 3}):
        issues.append((pid, f"unexpected label values: {label_classes}"))

print(f"Checked {len(patient_ids)} patients, found {len(issues)} issues")
for pid, problem in issues:
    print(pid, "-", problem)

In [ ]:
"""
Voxel-level class balance for the entire dataset based on patient_ids
"""

#0=background, 1=edema, 2=non-enhancing tumor, 3=enhancing tumour.
class_names = ["Background", "Oedema", "Non-Enhancing Tumour", "Enhancing Tumour"]
voxel_counts = np.zeros(4, dtype=np.int64)

sample_size = 50
sample_ids = patient_ids[:sample_size]
for pid in sample_ids:
    label = nib.load(label_files[pid]).get_fdata()
    values, counts = np.unique(label, return_counts=True)
    for v, c in zip(values, counts):
        voxel_counts[int(v)] += c

total_voxels = voxel_counts.sum()
percentages = 100 * voxel_counts / total_voxels

for name, count, pct in zip(class_names, voxel_counts, percentages):
    print(f"{name:30s} {count:>15,d} voxels  ({pct:6.3f}%)")

plt.bar(class_names, percentages)
plt.ylabel("% of voxels")
plt.title(f"Voxel-level class balance (sample of {sample_size} patients)")
plt.xticks(rotation=20)
plt.show()


### Train/val/test split

In [ ]:
from sklearn.model_selection import train_test_split

"""
Two-stage split at patient-level, so no patient's
scans ever appear in more than one of these three sets:
"""

# test set
train_val_ids, test_ids = train_test_split(
    patient_ids, test_size=0.15, random_state=42
)

# train/val set
train_ids, val_ids = train_test_split(
    train_val_ids, test_size=0.15/0.85, random_state=42
)

### Preprocessing Pipeline

In [ ]:
from monai.transforms import NormalizeIntensity
import numpy as np
import matplotlib.pyplot as plt

# channel_wise: separate mean/std per modality
normalise = NormalizeIntensity(nonzero=True, channel_wise=True)

raw_volume = nib.load(image_files[patient_ids[0]]).get_fdata()   # (H, W, D, 4)
volume_channel_first = np.moveaxis(raw_volume, -1, 0)             # (4, H, W, D)

normalised = normalise(volume_channel_first.copy())

# modality: 0=FLAIR, 1=T1w, 2=T1ce (t1gd), 3=T2w
modality_names = ["FLAIR", "T1", "T1ce", "T2"]
for i, name in enumerate(modality_names):
    before = volume_channel_first[i]
    after = normalised[i]
    print(f"{name:5s} before -> mean: {before[before > 0].mean():8.2f} std: {before[before > 0].std():8.2f}"
          f"   |   after -> mean: {after[after != 0].mean():6.3f} std: {after[after != 0].std():6.3f}")

# visualise one modality (FLAIR)
mod_idx = 0
before_mod = volume_channel_first[mod_idx]
after_mod = normalised[mod_idx]
slice_idx = before_mod.shape[-1] // 2

fig, axes = plt.subplots(2, 2, figsize=(10, 8))
axes[0, 0].hist(before_mod[before_mod > 0].ravel(), bins=100)
axes[0, 0].set_title(f"Before: {modality_names[mod_idx]} histogram")
axes[0, 1].hist(after_mod[after_mod != 0].ravel(), bins=100)
axes[0, 1].set_title(f"After: {modality_names[mod_idx]} histogram")
axes[1, 0].imshow(before_mod[..., slice_idx], cmap="gray")
axes[1, 0].set_title("Before: slice"); axes[1, 0].axis("off")
axes[1, 1].imshow(after_mod[..., slice_idx], cmap="gray")
axes[1, 1].set_title("After: slice"); axes[1, 1].axis("off")
plt.tight_layout()
plt.show()

### Dataset/Dataloader Design

In [ ]:
def make_data_dicts(ids):
    return [
        {"image": image_files[pid], "label": label_files[pid]}
        for pid in ids
    ]

train_files = make_data_dicts(train_ids)
val_files   = make_data_dicts(val_ids)
test_files  = make_data_dicts(test_ids)

# training: random patch crop + light augmentation
train_transforms = Compose([
    LoadImaged(keys=["image", "label"]),
    EnsureChannelFirstd(keys=["image"], channel_dim=-1),
    EnsureChannelFirstd(keys=["label"], channel_dim="no_channel"),
    NormalizeIntensityd(keys=["image"], channel_wise=True, nonzero=True),
    RandCropByPosNegLabeld(
        keys=["image", "label"],
        label_key="label",
        spatial_size=(96, 96, 96),
        pos=1, neg=1,
        num_samples=2,
    ),
    RandFlipd(keys=["image", "label"], spatial_axis=[0], prob=0.5),
    EnsureTyped(keys=["image", "label"]),
])

# val/test: no cropping or augmentation, deterministic
eval_transforms = Compose([
    LoadImaged(keys=["image", "label"]),
    EnsureChannelFirstd(keys=["image"], channel_dim=-1),
    EnsureChannelFirstd(keys=["label"], channel_dim="no_channel"),
    NormalizeIntensityd(keys=["image"], channel_wise=True, nonzero=True),
    EnsureTyped(keys=["image", "label"]),
])

train_ds = Dataset(data=train_files, transform=train_transforms)
val_ds   = Dataset(data=val_files,   transform=eval_transforms)
test_ds  = Dataset(data=test_files,  transform=eval_transforms)

train_loader = DataLoader(train_ds, batch_size=2, shuffle=True,  num_workers=8)
# batch_size=1: full-size volumes, one at a time via sliding-window inference
val_loader  = DataLoader(val_ds,  batch_size=1, shuffle=False, num_workers=8)
test_loader = DataLoader(test_ds, batch_size=1, shuffle=False, num_workers=8)

In [ ]:
print(type(train_loader), type(val_loader), type(test_loader))
batch = next(iter(train_loader))
print(batch["image"].shape, batch["label"].shape)

### CNN/Segmentation Model Implementation

#### Code: baseline 3D U-Net

In [ ]:
def build_unet():
    return UNet(
        spatial_dims=3,
        in_channels=4,  # modalities (FLAIR, T1, T1ce, T2)
        out_channels=4, # 4 classes coming out per voxel (background, oedema, non-enhancing tumour, enhancing tumour)
        channels=(16, 32, 64, 128, 256),
        strides=(2, 2, 2, 2),
        num_res_units=2,
    ).to(device)

baseline_model = build_unet()

# sanity check: one dummy batch through the model
with torch.no_grad():
    dummy_input = torch.zeros(1, 4, 96, 96, 96).to(device)
    dummy_output = baseline_model(dummy_input)
    print("Output shape:", dummy_output.shape)

#### Training loop

In [ ]:
def save_checkpoint(path, model, optimizer, scheduler, epoch, history, run_name):
    # checkpoint
    tmp = path + ".tmp"
    torch.save({
        "model": model.state_dict(),
        "optimizer": optimizer.state_dict(),
        "scheduler": scheduler.state_dict(),
        "epoch": epoch,
        "history": history,
        "run_name": run_name,
    }, tmp)
    os.replace(tmp, path)

def train_model(model, train_loader, val_loader, num_epochs, run_name,
                lr=1e-4, weight_decay=1e-5, lambda_ce=1.0, save_path=None):
    loss_function = DiceCELoss(to_onehot_y=True, softmax=True, lambda_ce=lambda_ce)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=num_epochs)

    history = {"train_loss": [], "val_loss": []}
    start_epoch = 0

    # resume if a checkpoint for this run already exists
    if save_path and os.path.exists(save_path):
        ckpt = torch.load(save_path, map_location=device)
        if ckpt["run_name"] != run_name:
            raise ValueError(f"{save_path} belongs to run '{ckpt['run_name']}', not '{run_name}'")
        model.load_state_dict(ckpt["model"])
        optimizer.load_state_dict(ckpt["optimizer"])
        scheduler.load_state_dict(ckpt["scheduler"])
        history = ckpt["history"]
        start_epoch = ckpt["epoch"]
        print(f"[{run_name}] resuming after epoch {start_epoch}/{num_epochs}")

    for epoch in range(start_epoch, num_epochs):
        model.train()
        epoch_loss = 0
        for batch in train_loader:
            inputs = batch["image"].to(device)
            labels = batch["label"].to(device)

            optimizer.zero_grad()
            outputs = model(inputs)
            loss = loss_function(outputs, labels)
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item()

        scheduler.step()
        avg_train_loss = epoch_loss / len(train_loader)
        history["train_loss"].append(avg_train_loss)

        # fast per-epoch check: val loss only
        model.eval()
        val_loss = 0
        with torch.no_grad():
            for batch in val_loader:
                inputs = batch["image"].to(device)
                labels = batch["label"].to(device)
                outputs = sliding_window_inference(inputs, (96, 96, 96), 4, model)
                val_loss += loss_function(outputs, labels).item()
        history["val_loss"].append(val_loss / len(val_loader))

        print(f"[{run_name}] epoch {epoch+1}/{num_epochs} "
              f"train_loss={avg_train_loss:.4f} val_loss={history['val_loss'][-1]:.4f}")

        # full checkpoint after every finished epoch
        if save_path:
            save_checkpoint(save_path, model, optimizer, scheduler, epoch + 1, history, run_name)

    return history


baseline_default_history = train_model(
    baseline_model, train_loader, val_loader,
    num_epochs=50, run_name="baseline_default", lr=1e-4,
)

In [ ]:
def to_regions(seg):
    # class ids 0-3 (dataset.json: 1 edema, 2 non-enhancing, 3 enhancing)
    # channels: WT = classes 1+2+3, TC = classes 2+3, ET = class 3
    return torch.cat([seg > 0, seg >= 2, seg == 3], dim=1).float()

def to_classes(seg):
    # channels: edema (1), non-enhancing (2), enhancing (3); background left out
    return torch.cat([seg == 1, seg == 2, seg == 3], dim=1).float()

def evaluate_model(model, loader):
    # include_background=True because these 3-channel masks have no background channel to skip
    region_dice = DiceMetric(include_background=True, reduction="mean_batch")
    class_dice = DiceMetric(include_background=True, reduction="mean_batch")

    model.eval()
    with torch.no_grad():
        for batch in loader:
            inputs = batch["image"].to(device)
            labels = batch["label"].to(device)
            outputs = sliding_window_inference(inputs, (96, 96, 96), 4, model)
            pred = torch.argmax(outputs, dim=1, keepdim=True)
            region_dice(y_pred=to_regions(pred), y=to_regions(labels))
            class_dice(y_pred=to_classes(pred), y=to_classes(labels))

    per_region = region_dice.aggregate()
    per_class = class_dice.aggregate()
    region_dice.reset()
    class_dice.reset()
    # per_region channels: 0=Whole Tumour (WT), 1=Tumour Core (TC), 2=Enhancing Tumour (ET)
    # per_class channels: 0=edema, 1=non-enhancing, 2=enhancing
    return {
        "mean_dice": per_region.mean().item(),   # drives tuning and model selection
        "per_region_dice": per_region.tolist(),
        "per_class_dice": per_class.tolist(),
    }

#### Hyperparameter tuning using RandoSearch

In [ ]:
TRIAL_EPOCHS = 5
TUNE_VAL_PATIENTS = 20
N_TRIALS = 20
SEARCH_SEED = 42
MAX_SEARCH_SECONDS = 9 * 3600
TRIALS_FILE = "trials.jsonl"

# fixed validation subset, so every trial is scored on the same patients
tune_val_loader = DataLoader(
    Dataset(data=val_files[:TUNE_VAL_PATIENTS], transform=eval_transforms),
    batch_size=1, shuffle=False, num_workers=8,
)

def log_uniform(rng, low, high):
    # equal chance for each factor-of-ten band between low and high
    return float(np.exp(rng.uniform(np.log(low), np.log(high))))

def sample_params(rng):
    return {
        "lr": log_uniform(rng, 1e-5, 1e-3),
        "weight_decay": log_uniform(rng, 1e-6, 1e-3),
        "lambda_ce": log_uniform(rng, 0.25, 2.0),
    }

# draw every trial's settings up front, so trial i has the same settings in every session
rng = np.random.default_rng(SEARCH_SEED)
all_params = [sample_params(rng) for _ in range(N_TRIALS)]

# anything that changes what a score means goes here; a mismatch stops the notebook
config = {"trial_epochs": TRIAL_EPOCHS, "tune_val_patients": TUNE_VAL_PATIENTS,
          "seed": SEARCH_SEED, "score": "mean Dice of WT/TC/ET"}

# load trials finished in an earlier session
trials = []
if os.path.exists(TRIALS_FILE):
    with open(TRIALS_FILE) as f:
        trials = [json.loads(line) for line in f if line.strip()]
    if any(t["config"] != config for t in trials):
        raise ValueError(f"{TRIALS_FILE} was made with different settings; delete it to start over")
done = {t["trial"] for t in trials}
print("trials already finished:", sorted(done))

def run_trial(i):
    params = all_params[i]
    start = time.time()
    trial_model = build_unet()
    train_model(
        trial_model, train_loader, tune_val_loader,
        num_epochs=TRIAL_EPOCHS, run_name=f"trial{i}", **params,
    )
    score = evaluate_model(trial_model, tune_val_loader)["mean_dice"]
    record = {"trial": i, "mean_dice": score, "params": params, "config": config}
    trials.append(record)
    with open(TRIALS_FILE, "a") as f:
        f.write(json.dumps(record) + "\n")
    print(f"trial {i} finished: mean_dice={score:.4f} in {time.time() - start:.0f} s, params={params}")

In [ ]:
search_start = time.time()
for i in range(N_TRIALS):
    if i in done:
        continue
    if time.time() - search_start > MAX_SEARCH_SECONDS:
        print(f"time budget reached, stopping with {len(trials)} of {N_TRIALS} trials finished")
        break
    run_trial(i)

best = max(trials, key=lambda t: t["mean_dice"])
best_params = best["params"]
print("best validation mean Dice:", round(best["mean_dice"], 4), "from", len(trials), "trials")
print("best settings:", best_params)

# top 5 trials: if the scores are close, the ranking between them is mostly noise
for t in sorted(trials, key=lambda t: t["mean_dice"], reverse=True)[:5]:
    print(t["trial"], round(t["mean_dice"], 4), t["params"])

# save the result so a notebook restart does not lose it
with open("best_params.json", "w") as f:
    json.dump(
        {"best_params": best_params, "best_mean_dice": best["mean_dice"],
         "n_trials_finished": len(trials), "n_trials_planned": N_TRIALS,
         "trial_epochs": TRIAL_EPOCHS, "tune_val_patients": TUNE_VAL_PATIENTS,
         "seed": SEARCH_SEED},
        f, indent=2,
    )

In [ ]:
# after a notebook restart: reload the saved result instead of re-running the search
with open("best_params.json") as f:
    best_params = json.load(f)["best_params"]
print("loaded best settings:", best_params)

In [ ]:
# fresh baseline trained once with the tuned settings
baseline_model = build_unet()
baseline_history = train_model(
    baseline_model, train_loader, val_loader,
    num_epochs=50, run_name="baseline", save_path="baseline_last.pt", **best_params
)